# Imports Needed

In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 16.9 MB/s eta 0:00:01
     ---------------- ----------------------- 5.2/12.8 MB 21.3 MB/s eta 0:00:01
     ---------------------- ----------------- 7.3/12.8 MB 18.1 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 15.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 16.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import spacy
import re
import matplotlib.pyplot as plt

# Loading Data

In [3]:
df = pd.read_csv("C:\\Users\\Carol\\AI-Hallucinations-Detection\\data\\cleaned_data.csv")

df.head()

,reference,input,output,label,hallucination_type_realized,question_type,hallucination_type_encouraged
0,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...","What organization has accredited A.D.A.M., Inc...","A.D.A.M., Inc. is accredited by the Atlantis H...",hallucinated,Entity-error hallucination,Default question type,Entity-error hallucination
1,"""This dataset for NOAA's Science On a Sphere d...",What type of educational activity is encourage...,The dataset encourages learners to complete a ...,hallucinated,Relation-error hallucination,Default question type,Relation-error hallucination
2,"Dimension items include ""Foreground"" and ""Back...",What is the primary reason for a hit to be cla...,The primary reason for a hit to be classified ...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination
3,"Atlas Search runs a new process, called mongot...",What are the specific hardware requirements fo...,"Based on our production monitoring data, mongo...",hallucinated,Unverifiable information hallucination,Other common hallucinated questions,Other hallucination
4,The business implications are stark. In a surv...,What percentage of banking executives in the l...,34% of banking executives in the loan originat...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination


# Named Entity and Numerical Fact Overlap Functions

| Feature | Low Value (close to 0) | High Value (close to 1) |
|----------|----------|----------|
| entity_recall | Response kept very few reference entities | Response kept most reference entities |
| out_of_reference_rate | Response introduced few new entities | Response introduced many new entities |
| number_overlap | Response changed or omitted most numbers | Response preserved most numbers |

In [4]:
#model we are using (reads people, organizations, locations, numbers, dates, etc.)
nlp = spacy.load("en_core_web_sm")

c:\Users\Carol\miniconda3\envs\spark-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


en_core_web_trf was taking quite awhile to run in our for loop when running the model with our data which we theorized that it was taking up too much GB in our RAM

In [5]:
#Entity Extraction function
def extract_entities(text):
    """Extract named entities from the given text using spaCy's NER model."""
    doc = nlp(text)

    entities = {}

    for ent in doc.ents:
        entities.setdefault(ent.label_, set()).add(ent.text.lower())
        
    return entities

In [6]:
#testing how the function works
sample = "Apple was founded by Steve Jobs in 1976."

extract_entities(sample)

{'ORG': {'apple'}, 'PERSON': {'steve jobs'}, 'DATE': {'1976'}}

In [7]:
#feature functions
def entity_recall(ref_entities, resp_entities):
    """Calculate how many entities from the reference appear in the response."""

    #create sets of entities for both reference and response
    ref_set = set()
    resp_set = set()

    for ents in ref_entities.values():
        ref_set.update(ents)

    for ents in resp_entities.values():
        resp_set.update(ents)

    if len(ref_set) == 0:
        return 1.0

    return len(ref_set & resp_set) / len(ref_set) #the response preserved that amount of entities from the reference

In [8]:
#out of reference function
def out_of_reference_rate(ref_entities, resp_entities):
    """Calculate how many entities in the response are not in the reference."""

    ref_set = set()
    resp_set = set()

    for ents in ref_entities.values():
        ref_set.update(ents)

    for ents in resp_entities.values():
        resp_set.update(ents)

    if len(resp_set) == 0:
        return 1.0

    return len(resp_set - ref_set) / len(resp_set) #the response has that amount of entities that are not in the reference

In [9]:
#Extract number of text
def extract_numbers(text):
    """Extract numbers from the given text using regular expressions."""
    
    return set(re.findall(r'\d+(?:\.\d+)?', str(text))) 

In [10]:
#numerical overlap function
def number_overlap(reference, response):
    """Calculate the overlap of numbers between the reference and response."""
    
    ref_nums = extract_numbers(reference)
    resp_nums = extract_numbers(response)

    if len(ref_nums) == 0:
        return 1.0

    return len(ref_nums & resp_nums) / len(ref_nums)

# Applying model to our data

In [11]:
print(df.columns)

Index(['reference', 'input', 'output', 'label', 'hallucination_type_realized',
       'question_type', 'hallucination_type_encouraged'],
      dtype='object')


In [12]:
reference_col = "reference"
response_col = "output"

In [13]:
features = []

#iterate through each row in the dataframe and calculate features
for _, row in df.iterrows():

    #for each row, get the reference and response text
    ref = row[reference_col] 
    resp = row[response_col]

    #extract entities from both reference and response
    ref_ents = extract_entities(ref)
    resp_ents = extract_entities(resp)

    features.append({
        "entity_recall":
            entity_recall(ref_ents, resp_ents),

        "out_of_reference_rate":
            out_of_reference_rate(ref_ents, resp_ents),

        "number_overlap":
            number_overlap(ref, resp)
    })

In [14]:
feature_df = pd.DataFrame(features)

feature_df.head()

,entity_recall,out_of_reference_rate,number_overlap
0,0.000000,1.000000,0.0
1,0.000000,1.000000,1.0
2,0.500000,0.000000,1.0
3,0.250000,0.900000,1.0
4,0.333333,0.666667,0.5


In [15]:
#To add the label column to the feature dataframe to see what the features look like for hallucinated vs non-hallucinated examples
feature_df["label"] = df["label"]

feature_df.head()

,entity_recall,out_of_reference_rate,number_overlap,label
0,0.000000,1.000000,0.0,hallucinated
1,0.000000,1.000000,1.0,hallucinated
2,0.500000,0.000000,1.0,hallucinated
3,0.250000,0.900000,1.0,hallucinated
4,0.333333,0.666667,0.5,hallucinated


# Observations

In [16]:
feature_df.groupby("label").mean(numeric_only=True)

,entity_recall,out_of_reference_rate,number_overlap
label,,,
factual,0.351255,0.561900,0.707139
hallucinated,0.388680,0.743071,0.782251


In our dataset, hallucinated responses showcases higher average entity recall, out-of-reference entity rates, and higher numerical overlap than factual responses. This could mean that our hallucinated responses can still repeat many correct entities and numbers while at the same time adding false information.